In [ ]:
import time
import requests
import pandas as pd
import numpy as np

BASE_URL = "https://wcc.sc.egov.usda.gov/awdbRestApi/services/v1"
ELEMENTS = ["WTEQ", "SNWD", "PREC", "TAVG", "TMIN", "TMAX"]


def get_snotel_stations():
    params = {
        "stationTriplets": "*:*:SNTL",
        "elements": "WTEQ",
        "activeOnly": "false",
    }

    r = requests.get(f"{BASE_URL}/stations", params=params, timeout=60)
    r.raise_for_status()

    stations = pd.DataFrame(r.json())
    stations = stations[stations["networkCode"] == "SNTL"].copy()

    return stations


def parse_awdb_response(data):
    rows = []

    for station_obj in data:
        station_triplet = station_obj.get("stationTriplet")

        for element_obj in station_obj.get("data", []):
            station_element = element_obj.get("stationElement", {})
            element_code = station_element.get("elementCode")

            for v in element_obj.get("values", []):
                rows.append({
                    "station_triplet": station_triplet,
                    "date": v.get("date"),
                    element_code: v.get("value"),
                })

    df = pd.DataFrame(rows)

    if df.empty:
        return df

    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    df = (
        df.groupby(["station_triplet", "date"], as_index=False)
          .first()
    )

    return df


def get_station_daily_data(station_triplet, start_date, end_date):
    params = {
        "stationTriplets": station_triplet,
        "elements": ",".join(ELEMENTS),
        "duration": "daily",
        "beginDate": start_date,
        "endDate": end_date,
    }

    r = requests.get(f"{BASE_URL}/data", params=params, timeout=60)
    r.raise_for_status()

    return parse_awdb_response(r.json())


def download_all_daily_data(stations, start_date="2000-01-01", end_date="2025-12-31", max_stations=None):
    station_list = stations["stationTriplet"].dropna().unique()

    if max_stations is not None:
        station_list = station_list[:max_stations]

    all_data = []

    for i, station in enumerate(station_list, start=1):
        try:
            print(f"{i}/{len(station_list)} downloading {station}")

            df = get_station_daily_data(
                station_triplet=station,
                start_date=start_date,
                end_date=end_date,
            )

            if not df.empty:
                all_data.append(df)

            time.sleep(0.2)

        except Exception as e:
            print(f"Failed for {station}: {e}")

    if not all_data:
        return pd.DataFrame()

    return pd.concat(all_data, ignore_index=True)


def clean_daily_data(daily):
    daily = daily.copy()

    for col in ELEMENTS:
        if col not in daily.columns:
            daily[col] = np.nan
        daily[col] = pd.to_numeric(daily[col], errors="coerce")

    daily = daily.sort_values(["station_triplet", "date"])

    daily["daily_precip"] = (
        daily.groupby("station_triplet")["PREC"]
        .diff()
    )

    daily.loc[daily["daily_precip"] < 0, "daily_precip"] = np.nan

    return daily


def build_monthly_dataset(daily, stations):
    daily = clean_daily_data(daily)

    monthly = (
        daily
        .set_index("date")
        .groupby("station_triplet")
        .resample("ME")
        .agg({
            "WTEQ": ["last", "mean", "min", "max"],
            "SNWD": ["last", "mean", "min", "max"],
            "daily_precip": "sum",
            "TAVG": ["mean", "min", "max", "std"],
            "TMIN": ["mean", "min"],
            "TMAX": ["mean", "max"],
        })
    )

    monthly.columns = ["_".join(col).strip() for col in monthly.columns]
    monthly = monthly.reset_index()

    monthly = monthly.rename(columns={
        "date": "month_end",
        "WTEQ_last": "current_swe",
        "SNWD_last": "current_snow_depth",
        "daily_precip_sum": "monthly_precip",
    })

    monthly["month"] = monthly["month_end"].dt.month
    monthly["year"] = monthly["month_end"].dt.year
    monthly["day_of_year"] = monthly["month_end"].dt.dayofyear

    monthly["month_sin"] = np.sin(2 * np.pi * monthly["month"] / 12)
    monthly["month_cos"] = np.cos(2 * np.pi * monthly["month"] / 12)

    monthly["is_winter"] = monthly["month"].isin([12, 1, 2]).astype(int)
    monthly["is_spring"] = monthly["month"].isin([3, 4, 5]).astype(int)

    monthly["water_year"] = monthly["year"] + (monthly["month"] >= 10).astype(int)

    monthly = monthly.sort_values(["station_triplet", "month_end"])
    group = monthly.groupby("station_triplet")

    monthly["swe_lag_1"] = group["current_swe"].shift(1)
    monthly["swe_lag_2"] = group["current_swe"].shift(2)
    monthly["swe_lag_3"] = group["current_swe"].shift(3)

    monthly["snow_depth_lag_1"] = group["current_snow_depth"].shift(1)

    monthly["precip_lag_1"] = group["monthly_precip"].shift(1)
    monthly["precip_lag_2"] = group["monthly_precip"].shift(2)

    monthly["precip_rolling_3mo"] = (
        group["monthly_precip"]
        .rolling(3)
        .sum()
        .reset_index(level=0, drop=True)
    )

    monthly["swe_change_1mo"] = monthly["current_swe"] - monthly["swe_lag_1"]
    monthly["swe_change_2mo"] = monthly["current_swe"] - monthly["swe_lag_2"]

    monthly["water_year_precip_to_date"] = (
        monthly.groupby(["station_triplet", "water_year"])["monthly_precip"]
        .cumsum()
    )

    monthly["water_year_max_swe_so_far"] = (
        monthly.groupby(["station_triplet", "water_year"])["current_swe"]
        .cummax()
    )

    monthly["avg_temp_above_freezing"] = (monthly["TAVG_mean"] > 32).astype(int)

    monthly["freeze_thaw_risk"] = (
        (monthly["TMIN_mean"] < 32) &
        (monthly["TMAX_mean"] > 32)
    ).astype(int)

    monthly["target_swe_next_month"] = group["current_swe"].shift(-1)
    monthly["target_month_end"] = group["month_end"].shift(-1)

    station_features = stations.rename(columns={
        "stationTriplet": "station_triplet"
    }).copy()

    keep_cols = [
        "station_triplet",
        "name",
        "stateCode",
        "countyName",
        "huc",
        "elevation",
        "latitude",
        "longitude",
        "beginDate",
        "endDate",
    ]

    keep_cols = [col for col in keep_cols if col in station_features.columns]

    station_features = (
        station_features[keep_cols]
        .drop_duplicates("station_triplet")
    )

    monthly = monthly.merge(
        station_features,
        on="station_triplet",
        how="left"
    )

    monthly = monthly.dropna(subset=[
        "current_swe",
        "target_swe_next_month",
        "swe_lag_1",
        "swe_lag_2",
    ])

    return monthly


stations = get_snotel_stations()

daily = download_all_daily_data(
    stations,
    start_date="2000-01-01",
    end_date="2025-12-31",
    max_stations=None,
)

daily.to_csv("../raw_data/snotel_daily_raw.csv", index=False)

monthly = build_monthly_dataset(daily, stations)

monthly.to_csv("../raw_data/snotel_monthly_swe_prediction.csv", index=False)

print(daily.shape)
print(monthly.shape)

monthly.head()

1/891 downloading 301:CA:SNTL
2/891 downloading 907:UT:SNTL
3/891 downloading 916:MT:SNTL
4/891 downloading 1267:AK:SNTL
5/891 downloading 908:WA:SNTL
6/891 downloading 1344:CO:SNTL
7/891 downloading 1189:AK:SNTL
8/891 downloading 1062:AK:SNTL
9/891 downloading 1070:AK:SNTL
10/891 downloading 302:OR:SNTL
11/891 downloading 2065:AK:SNTL
12/891 downloading 1000:OR:SNTL
13/891 downloading 303:CO:SNTL
14/891 downloading 1030:CO:SNTL
15/891 downloading 304:OR:SNTL
16/891 downloading 305:CO:SNTL
17/891 downloading 306:ID:SNTL
18/891 downloading 1308:UT:SNTL
19/891 downloading 307:MT:SNTL
20/891 downloading 308:AZ:SNTL
21/891 downloading 1140:AZ:SNTL
22/891 downloading 309:WY:SNTL
23/891 downloading 310:AZ:SNTL
24/891 downloading 311:MT:SNTL
25/891 downloading 312:ID:SNTL
26/891 downloading 1212:AZ:SNTL
27/891 downloading 313:MT:SNTL
28/891 downloading 314:WY:SNTL
29/891 downloading 315:MT:SNTL
30/891 downloading 1190:MT:SNTL
31/891 downloading 316:NM:SNTL
32/891 downloading 317:WY:SNTL
33/89

,station_triplet,month_end,current_swe,WTEQ_mean,WTEQ_min,WTEQ_max,current_snow_depth,SNWD_mean,SNWD_min,SNWD_max,...,target_month_end,name,stateCode,countyName,huc,elevation,latitude,longitude,beginDate,endDate
2,1000:OR:SNTL,2000-11-30,3.0,1.283333,0.2,3.0,14.0,5.600000,0.0,14.0,...,2000-12-31,Annie Springs,OR,Klamath,180102030101,6020.0,42.87007,-122.16518,2000-09-01 00:00,2100-01-01 00:00
3,1000:OR:SNTL,2000-12-31,9.0,6.051613,2.9,9.0,33.0,25.258065,8.0,41.0,...,2001-01-31,Annie Springs,OR,Klamath,180102030101,6020.0,42.87007,-122.16518,2000-09-01 00:00,2100-01-01 00:00
4,1000:OR:SNTL,2001-01-31,12.5,10.525806,9.0,12.5,45.0,37.935484,30.0,47.0,...,2001-02-28,Annie Springs,OR,Klamath,180102030101,6020.0,42.87007,-122.16518,2000-09-01 00:00,2100-01-01 00:00
5,1000:OR:SNTL,2001-02-28,16.5,14.439286,12.5,16.5,52.0,45.571429,36.0,59.0,...,2001-03-31,Annie Springs,OR,Klamath,180102030101,6020.0,42.87007,-122.16518,2000-09-01 00:00,2100-01-01 00:00
6,1000:OR:SNTL,2001-03-31,16.0,17.645161,16.0,18.8,38.0,49.709677,38.0,60.0,...,2001-04-30,Annie Springs,OR,Klamath,180102030101,6020.0,42.87007,-122.16518,2000-09-01 00:00,2100-01-01 00:00
